# SIGNBRIDGE — free ASL training pilot (Kaggle)

This notebook uses Kaggle's free notebook compute when available. It does not create paid cloud resources. Kaggle GPU access is subject to availability and account quotas.

**Scope:** downloads only Épée keypoints/annotations/metadata (no RGB videos), converts them to landmark features, then runs a short CTC baseline pilot. This is not a sentence translator or live-call interpreter.

**Before running:** In Kaggle Notebook Settings, enable **Internet** and select **GPU** if one is available. Run cells in order. The dataset authorization flag records the user's attestation; it does not manufacture written permission or override dataset terms.


In [ ]:
import os, sys, json, subprocess, pathlib, torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU assigned. You can still run the data audit/feature conversion on CPU; training will be slower.')


In [ ]:
%pip -q install --upgrade huggingface_hub
print('Hugging Face Hub ready')


In [ ]:
from huggingface_hub import hf_hub_download
import csv, collections, re
metadata_file = pathlib.Path(hf_hub_download(repo_id='CLERC-DATA/epee', repo_type='dataset', filename='metadata.csv'))
with metadata_file.open(newline='', encoding='utf-8') as handle:
    metadata_rows = list(csv.DictReader(handle))
train_signers = {'ALPHA', 'BRAVO', 'CHARLIE', 'FOXTROT'}
validation_signers = {'DELTA', 'GOLF'}
test_signers = {'ECHO', 'HOTEL'}
split_groups = {'train': train_signers, 'validation': validation_signers, 'test': test_signers}
split_rows = {name: [r for r in metadata_rows if r.get('signer_id', '').strip() in signers] for name, signers in split_groups.items()}
assert all(split_rows.values()), 'At least one signer split has no metadata rows; review signer IDs before downloading keypoints.'
assert len(metadata_rows) == 2400, f'Expected 2,400 audited metadata rows, found {len(metadata_rows)}.'
clip_ids = [r.get('clip_id', '').strip() for r in metadata_rows]
assert all(clip_ids) and len(set(clip_ids)) == len(clip_ids), 'Metadata clip IDs must be non-empty and unique.'
expected_split_counts = {'train': 1200, 'validation': 600, 'test': 600}
actual_split_counts = {name: len(group) for name, group in split_rows.items()}
assert actual_split_counts == expected_split_counts, f'Signer split counts differ from the audited plan: {actual_split_counts}'
split_signer_sets = {name: {r.get('signer_id', '').strip() for r in group} for name, group in split_rows.items()}
assert not (split_signer_sets['train'] & split_signer_sets['validation'] or split_signer_sets['train'] & split_signer_sets['test'] or split_signer_sets['validation'] & split_signer_sets['test']), 'Signer leakage detected across splits.'
def gloss_tokens(row):
    raw = re.sub(r'\s+', ' ', row.get('gloss_sequence', '').replace('|', ' ').strip().upper())
    return raw.split()
train_vocab = {token for row in split_rows['train'] for token in gloss_tokens(row)}
unseen = sorted({token for name in ('validation', 'test') for row in split_rows[name] for token in gloss_tokens(row)} - train_vocab)
def minimum_ctc_frames(tokens):
    return len(tokens) + sum(a == b for a, b in zip(tokens, tokens[1:]))

too_short = [r.get('clip_id', '?') for group in split_rows.values() for r in group if minimum_ctc_frames(gloss_tokens(r)) > int(r.get('n_frames', 0) or 0)]
phrase_sets = {name: {r.get('phrase_id', '').strip() for r in group} for name, group in split_rows.items()}
phrase_overlap = sorted((phrase_sets['train'] & phrase_sets['validation']) | (phrase_sets['train'] & phrase_sets['test']))
print(json.dumps({'metadata_rows': len(metadata_rows), 'split_counts': {k: len(v) for k,v in split_rows.items()}, 'training_gloss_vocabulary': len(train_vocab), 'unseen_heldout_gloss_tokens': unseen[:40], 'unseen_token_count': len(unseen), 'too_short_clip_count': len(too_short), 'phrase_overlap_count': len(phrase_overlap)}, indent=2))
if unseen:
    raise RuntimeError('Preflight stopped before the large keypoint download: held-out gloss tokens are absent from the training vocabulary. Review the signer split/labels; do not leak held-out signer clips into training.')
if too_short:
    raise RuntimeError(f'Preflight stopped: {len(too_short)} clips have fewer frames than gloss tokens; inspect annotation/frame alignment before downloading keypoints.')
print('Metadata preflight passed. Proceeding to keypoint download is now reasonable.')


In [ ]:
REPO = pathlib.Path('/kaggle/working/Signbridge')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/abhiparakh123-source/Signbridge-.git', str(REPO)], check=True)
TRAIN = REPO / 'services/isl-stream/training'
assert (TRAIN / 'download_epee_keypoints.py').is_file(), 'Training scripts not found in the cloned repository'
print('Using repository:', REPO)


In [ ]:
DATA = pathlib.Path('/kaggle/working/epee-asl')
REPORT = pathlib.Path('/kaggle/working/download-report.json')
subprocess.run([
    sys.executable, str(TRAIN / 'download_epee_keypoints.py'),
    '--destination', str(DATA),
    '--user-attests-rights',
    '--report', str(REPORT),
], check=True)
download_report = json.loads(REPORT.read_text())
print(json.dumps(download_report, indent=2))
assert download_report['ok'] and download_report['keypoint_files'] == 2400 and download_report['annotation_files'] == 2400
assert download_report['videos_downloaded'] is False


In [ ]:
FEATURES = pathlib.Path('/kaggle/working/epee-asl-features')
subprocess.run([
    sys.executable, str(TRAIN / 'prepare_epee_keypoints.py'),
    '--dataset-root', str(DATA), '--output-root', str(FEATURES),
    '--train-signers', 'ALPHA', 'BRAVO', 'CHARLIE', 'FOXTROT',
    '--validation-signers', 'DELTA', 'GOLF',
    '--test-signers', 'ECHO', 'HOTEL',
    '--user-attests-rights',
], check=True)
manifest = FEATURES / 'features_manifest.csv'
assert manifest.is_file()
print('Feature manifest:', manifest)


## Short training pilot

This starts with **3 epochs** to verify the full training path within free-session limits. It is an experiment, not a production-quality model. The script evaluates on signer-held-out validation and test sets and saves the best checkpoint plus metrics.


In [ ]:
OUT = pathlib.Path('/kaggle/working/signbridge-asl-pilot/best.pt')
subprocess.run([
    sys.executable, str(TRAIN / 'train_ctc.py'),
    '--manifest', str(manifest), '--data-root', str(FEATURES),
    '--language', 'ASL', '--output', str(OUT),
    '--epochs', '3', '--batch-size', '8', '--seed', '7', '--causal',
], check=True)
metrics_path = OUT.with_suffix('.metrics.json')
assert OUT.is_file() and metrics_path.is_file()
metrics = json.loads(metrics_path.read_text())
print(json.dumps({'checkpoint': str(OUT), 'test_metrics': metrics['test_metrics'], 'device': metrics['device']}, indent=2))


In [ ]:
import hashlib, shutil
artifact_dir = pathlib.Path('/kaggle/working/signbridge-artifacts')
artifact_dir.mkdir(exist_ok=True)
shutil.copy2(OUT, artifact_dir / 'signbridge-asl-pilot.pt')
shutil.copy2(metrics_path, artifact_dir / 'signbridge-asl-pilot.metrics.json')
digest = hashlib.sha256((artifact_dir / 'signbridge-asl-pilot.pt').read_bytes()).hexdigest()
summary = {
    'status': 'pilot_trained', 'language': 'ASL', 'dataset': download_report['dataset'],
    'clips_downloaded': download_report['keypoint_files'], 'checkpoint_sha256': digest,
    'device': metrics['device'], 'test_metrics': metrics['test_metrics'],
    'production_ready': False,
    'limitations': ['gloss baseline only', 'not sentence translation', 'not connected to live camera inference', 'not phone-call interpretation']
}
(artifact_dir / 'pilot-summary.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print('\nArtifacts are in /kaggle/working/signbridge-artifacts. Use Kaggle Save Version / Output to retain them.')
